# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*



I built the feature vector using only information that can be observed before the prediction/decision point.

The preprocessing steps are:

- Separate the target from the input features.
- Keep numeric features as numeric.
- Fill missing numeric values with the median.
- Convert categorical features into one-hot encoded columns.
- Fill missing categorical values with `"Unknown"`.
- Keep the preprocessing steps inside a reproducible pipeline.

This produces a model-ready feature matrix without using the target itself as an input feature.

In [3]:
import os

print("Current notebook folder:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir())

Current notebook folder:
d:\Internship\Week1\Task1-Week1\work\notebooks

Files in this folder:
['capstone.ipynb', 'w01_research_question.ipynb', 'w02_ml_task_framing.ipynb', 'w03_data_contract.ipynb', 'w03_feature_leakage_check.ipynb', 'w04_baseline_score.ipynb', 'w04_signal_audit.ipynb', 'w05_model.ipynb', 'w06_validation_audit.ipynb', 'w07_action_playbook.ipynb']


In [4]:
import os

matches = []

for root, dirs, files in os.walk(".."):
    for file in files:
        if file.lower() == "baseline_action_score.csv":
            matches.append(os.path.abspath(os.path.join(root, file)))

print("Found files:")
for path in matches:
    print(path)

Found files:
d:\Internship\Week1\Task1-Week1\work\outputs\baseline_action_score.csv


In [5]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------

DATA_PATH = r"D:\Internship\Week1\Task1-Week1\work\outputs\baseline_action_score.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (30000, 47)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'score', 'reason_code', 'action']

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
0,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,0.84,24.11,0.0,excellent,striking,down,-85.6,61678,STALE_HIGH_IMPRESSIONS,Refresh Content
1,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,3.66,42.99,0.0,excellent,page_3_5,down,-81.5,59472,STALE_HIGH_IMPRESSIONS,Refresh Content
2,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.0,good,page_3_5,down,-74.7,25715,STALE_HIGH_IMPRESSIONS,Refresh Content
3,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3478.0,21948.0,...,5.13,41.76,0.0,good,striking,down,-51.8,13299,STALE_HIGH_IMPRESSIONS,Refresh Content
4,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,0.00,40.00,0.0,good,page_3_5,down,-89.1,7812,STALE_HIGH_IMPRESSIONS,Refresh Content


In [6]:
# Check each column
for col in df.columns:
    print(f"{col} -> {df[col].dtype}")

content_id -> object
client_id -> object
search_volume -> float64
competition -> float64
competition_level -> object
cpc -> float64
content_type -> object
main_intent -> object
word_count -> float64
char_count -> float64
provider_used -> object
model_used -> object
impressions_90d -> int64
clicks_90d -> int64
pageviews_90d -> int64
sessions_90d -> int64
users_90d -> int64
engaged_sessions_90d -> int64
ai_sessions_90d -> int64
scroll_events_90d -> int64
days_with_impressions -> int64
days_with_sessions -> int64
impressions_last_30d -> int64
clicks_last_30d -> int64
sessions_last_30d -> int64
impressions_prev_30d -> int64
clicks_prev_30d -> int64
sessions_prev_30d -> int64
content_age_days -> int64
age_tier -> object
age_tier_order -> int64
days_since_last_update -> int64
freshness_tier -> object
word_count_tier -> object
char_count_tier -> object
ctr -> float64
avg_position -> float64
engagement_rate -> float64
scroll_rate -> float64
ai_traffic_pct -> float64
impression_tier -> object
p

In [7]:
print("Missing values:")
display(df.isna().sum())

print("\nNumber of unique values:")
display(df.nunique())

Missing values:


content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier                      0
age_tier_order                0
days_sin


Number of unique values:


content_id                30000
client_id                    32
search_volume                41
competition                 101
competition_level             3
cpc                         915
content_type                  3
main_intent                   4
word_count                 5476
char_count                14839
provider_used                 2
model_used                    5
impressions_90d            9438
clicks_90d                  477
pageviews_90d               856
sessions_90d                666
users_90d                   644
engaged_sessions_90d         68
ai_sessions_90d              35
scroll_events_90d           155
days_with_impressions        88
days_with_sessions           90
impressions_last_30d       5182
clicks_last_30d             239
sessions_last_30d           359
impressions_prev_30d       5931
clicks_prev_30d             258
sessions_prev_30d           311
content_age_days            225
age_tier                      4
age_tier_order                4
days_sin

In [8]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# =========================================================
# 1. Load data
# =========================================================

DATA_PATH = r"D:\Internship\Week1\Task1-Week1\work\outputs\baseline_action_score.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)


# =========================================================
# 2. Fields excluded before feature engineering
# =========================================================

EXCLUDED_COLUMNS = [
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action"
]


# Keep only columns that actually exist
excluded_existing = [
    col for col in EXCLUDED_COLUMNS
    if col in df.columns
]

print("\nExcluded columns:")
print(excluded_existing)


# =========================================================
# 3. Build candidate feature table
# =========================================================

X = df.drop(columns=excluded_existing)

print("\nCandidate feature shape:", X.shape)


# =========================================================
# 4. Identify numeric and categorical features
# =========================================================

numeric_features = X.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()


print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


# =========================================================
# 5. Numeric preprocessing
# =========================================================

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# =========================================================
# 6. Categorical preprocessing
# =========================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# =========================================================
# 7. Combine preprocessing
# =========================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)


# =========================================================
# 8. Build feature vector
# =========================================================

X_vector = preprocessor.fit_transform(X)


# =========================================================
# 9. Show result
# =========================================================

print("\nOriginal candidate features:", X.shape[1])
print("Final feature-vector shape:", X_vector.shape)
print("Rows:", X_vector.shape[0])
print("Model-ready feature count:", X_vector.shape[1])

Dataset shape: (30000, 47)

Excluded columns:
['content_id', 'client_id', 'score', 'reason_code', 'action']

Candidate feature shape: (30000, 42)

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']

Categorical features:
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'trend_direction']

Original candidate features: 42
Final feature-vector shape: (30000

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*



The feature vector contains 42 candidate features: 30 numeric and 12 categorical.

Numeric features are kept as numeric values. Missing numeric values are filled with the median during preprocessing.

Categorical features are filled with `"Unknown"` when missing and then one-hot encoded. Unknown categories at prediction time are ignored by the encoder.

I excluded identifiers (`content_id`, `client_id`) because they identify records rather than describe the content decision context. I also excluded `score`, `reason_code`, and `action` because these are outputs of the existing baseline decision logic rather than independent input features.

The key availability rule is:

> A feature is valid only if its value would be available at the moment the decision is made.

Historical performance windows such as 90-day and previous-30-day metrics are treated as available inputs only if they represent information already observed before that decision point.

In [9]:
feature_notes = []

for col in X.columns:

    if col in numeric_features:
        feature_type = "Numeric"
        missing_handling = "Median imputation"

    else:
        feature_type = "Categorical"
        missing_handling = 'Fill missing values with "Unknown"'

    feature_notes.append({
        "feature": col,
        "type": feature_type,
        "missing_handling": missing_handling,
        "available_before_prediction": "Requires timing check"
    })

feature_notes_df = pd.DataFrame(feature_notes)

display(feature_notes_df)

,feature,type,missing_handling,available_before_prediction
0,search_volume,Numeric,Median imputation,Requires timing check
1,competition,Numeric,Median imputation,Requires timing check
2,competition_level,Categorical,"Fill missing values with ""Unknown""",Requires timing check
3,cpc,Numeric,Median imputation,Requires timing check
4,content_type,Categorical,"Fill missing values with ""Unknown""",Requires timing check
5,main_intent,Categorical,"Fill missing values with ""Unknown""",Requires timing check
6,word_count,Numeric,Median imputation,Requires timing check
7,char_count,Numeric,Median imputation,Requires timing check
8,provider_used,Categorical,"Fill missing values with ""Unknown""",Requires timing check
9,model_used,Categorical,"Fill missing values with ""Unknown""",Requires timing check


In [10]:
# =========================================================
# Feature availability notes
# =========================================================

availability_map = {}

# Content / keyword attributes
for col in [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "provider_used",
    "model_used"
]:
    if col in X.columns:
        availability_map[col] = "Yes - available in content snapshot"


# Historical performance metrics
for col in [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]:
    if col in X.columns:
        availability_map[col] = "Yes - historical window, if snapshot precedes decision"


# Age / freshness features
for col in [
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier"
]:
    if col in X.columns:
        availability_map[col] = "Yes - based on content age/freshness at snapshot"


# Content-size derived features
for col in [
    "word_count_tier",
    "char_count_tier"
]:
    if col in X.columns:
        availability_map[col] = "Yes - derived from content size"


# Performance-derived metrics
for col in [
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier",
    "trend_direction",
    "trend_pct"
]:
    if col in X.columns:
        availability_map[col] = "Yes - only if calculated from data available at snapshot"


# Build final notes table
feature_notes_final = []

for col in X.columns:

    if col in numeric_features:
        feature_type = "Numeric"
        missing_handling = "Median imputation"
    else:
        feature_type = "Categorical"
        missing_handling = 'Fill missing values with "Unknown"'

    feature_notes_final.append({
        "feature": col,
        "type": feature_type,
        "missing_handling": missing_handling,
        "available_when": availability_map.get(
            col,
            "Timing must be verified"
        )
    })

feature_notes_final_df = pd.DataFrame(feature_notes_final)

display(feature_notes_final_df)

,feature,type,missing_handling,available_when
0,search_volume,Numeric,Median imputation,Yes - available in content snapshot
1,competition,Numeric,Median imputation,Yes - available in content snapshot
2,competition_level,Categorical,"Fill missing values with ""Unknown""",Yes - available in content snapshot
3,cpc,Numeric,Median imputation,Yes - available in content snapshot
4,content_type,Categorical,"Fill missing values with ""Unknown""",Yes - available in content snapshot
5,main_intent,Categorical,"Fill missing values with ""Unknown""",Yes - available in content snapshot
6,word_count,Numeric,Median imputation,Yes - available in content snapshot
7,char_count,Numeric,Median imputation,Yes - available in content snapshot
8,provider_used,Categorical,"Fill missing values with ""Unknown""",Yes - available in content snapshot
9,model_used,Categorical,"Fill missing values with ""Unknown""",Yes - available in content snapshot


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [11]:
# =========================================================
# Leakage Hunt - Test 2
# Scan for suspicious feature names
# =========================================================

leakage_terms = [
    "target",
    "label",
    "outcome",
    "future",
    "post",
    "after",
    "result",
    "converted",
    "conversion",
    "final"
]

suspicious_features = []

for col in X.columns:
    col_lower = col.lower()

    if any(term in col_lower for term in leakage_terms):
        suspicious_features.append(col)

print("Suspicious feature names:")
print(suspicious_features)

if len(suspicious_features) == 0:
    print("\nPASS: No obvious target/future/post-decision names found.")
else:
    print("\nREVIEW REQUIRED: These columns need manual timing review.")

Suspicious feature names:
[]

PASS: No obvious target/future/post-decision names found.


In [12]:
# =========================================================
# Leakage Hunt - Test 3
# Identify derived / ratio / tier features
# =========================================================

derived_features = [
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct",
    "competition_level",
    "age_tier",
    "age_tier_order",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "trend_direction"
]

derived_present = [
    col for col in derived_features
    if col in X.columns
]

print("Derived or transformed features found:")
for col in derived_present:
    print("-", col)

print("\nThese features require one rule:")
print(
    "They are valid only when calculated entirely from information "
    "available at or before the prediction snapshot."
)

Derived or transformed features found:
- ctr
- engagement_rate
- scroll_rate
- ai_traffic_pct
- trend_pct
- competition_level
- age_tier
- age_tier_order
- freshness_tier
- word_count_tier
- char_count_tier
- impression_tier
- position_tier
- trend_direction

These features require one rule:
They are valid only when calculated entirely from information available at or before the prediction snapshot.


In [13]:
# =========================================================
# Leakage Hunt - Test 4
# Confirm baseline decision outputs are excluded
# =========================================================

baseline_outputs = [
    "score",
    "reason_code",
    "action"
]

print("Baseline decision outputs in original data:")

for col in baseline_outputs:
    if col in df.columns:
        print(f"- {col}: present in original dataset")

print("\nBaseline decision outputs in feature set:")

remaining = [
    col for col in baseline_outputs
    if col in X.columns
]

print(remaining)

assert len(remaining) == 0

print("\nPASS: score, reason_code, and action are excluded.")

Baseline decision outputs in original data:
- score: present in original dataset
- reason_code: present in original dataset
- action: present in original dataset

Baseline decision outputs in feature set:
[]

PASS: score, reason_code, and action are excluded.


### Leakage test conclusion

I tested the feature set for obvious target, future, post-decision, and outcome-related column names.

No obvious suspicious names were found in the retained feature set.

I also reviewed derived features such as CTR, engagement rate, scroll rate, traffic percentages, trend features, and tier variables. These are not automatically leakage, but they are valid only when calculated from information available at or before the prediction snapshot.

The baseline outputs `score`, `reason_code`, and `action` were explicitly excluded. They are decision outputs rather than independent input features.

Historical 30-day and 90-day features are treated as valid only when their measurement window ends before the prediction decision point.

Therefore, the feature vector is treated as leakage-aware for this snapshot, with timing of historical and derived metrics kept as an explicit assumption.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*



I excluded five fields from the feature vector.

| Field | Why I excluded it |
|---|---|
| `content_id` | It is an identifier for the content item, not a useful decision feature. |
| `client_id` | It identifies the client and is unnecessary for this feature vector; excluding it also reduces privacy and memorization risk. |
| `score` | It is an existing baseline decision score, so using it as an input would make the model depend on the existing decision logic. |
| `reason_code` | It explains the baseline decision and is derived from the decision logic, so it is not an independent input. |
| `action` | It is the baseline recommended action and would leak the decision into the feature vector. |

I kept content attributes, historical performance metrics, freshness information, and derived rates only when they can be justified as available at or before the prediction snapshot.

In [14]:
# =========================================================
# Exclusion audit
# =========================================================

excluded_columns = [
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action"
]

exclusion_audit = pd.DataFrame({
    "excluded_field": excluded_columns,
    "present_in_original_data": [
        col in df.columns for col in excluded_columns
    ],
    "present_in_feature_vector": [
        col in X.columns for col in excluded_columns
    ]
})

display(exclusion_audit)

# Confirm none of the excluded fields entered X
assert not any(
    exclusion_audit["present_in_feature_vector"]
)

print("PASS: all excluded fields are absent from the feature vector.")

,excluded_field,present_in_original_data,present_in_feature_vector
0,content_id,True,False
1,client_id,True,False
2,score,True,False
3,reason_code,True,False
4,action,True,False


PASS: all excluded fields are absent from the feature vector.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries are included in the notebook.
- [x] Claims use careful words such as observed, available, treated as, and decision-support.
- [ ] Committed to my repo under `work/notebooks/`.
- [ ] Repo URL has been submitted on the assignment card.